# Stacking & blending

Stacking learns how to combine base models using out-of-fold predictions as honest features.

Stacking and blending keep the validation-warning from learning theory front and center: the meta-model must learn from honest predictions, not from base learners evaluating their own training rows. The notebook uses real base estimators and a logistic meta-learner. Save a copy to Drive to edit.

In [ ]:

import math
import warnings

import matplotlib.pyplot as plt
import numpy as np
from sklearn.base import clone
from sklearn.datasets import load_breast_cancer
from sklearn.datasets import load_wine
from sklearn.datasets import make_blobs
from sklearn.datasets import make_moons
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore")
np.random.seed(7)

def clf_ladder():
    """D1..D5 classification ladder of rising complexity. Returns [(name, X, y), ...].

    All X are 2-D float feature matrices, y integer labels, so one classifier runs unchanged
    across every rung (the 'watch it scale' story). Rungs get harder: clean+separable -> real
    high-dimensional. D1 is hand-built and fully inspectable.
    """
    rungs = []

    # D1 — four hand-placed 2-D points, 2 classes, clearly separable.
    x1 = np.array([[0.0, 0.0], [0.4, 0.2], [3.0, 3.0], [2.6, 3.2]])
    y1 = np.array([0, 0, 1, 1])
    rungs.append(("D1 hand 2-D points", x1, y1))

    # D2 — clean, well-separated Gaussian blobs.
    x2, y2 = make_blobs(n_samples=200, centers=3, cluster_std=0.8, random_state=1)
    rungs.append(("D2 clean blobs (3-class)", x2, y2))

    # D3 — non-linear, overlapping two-moons with noise.
    x3, y3 = make_moons(n_samples=300, noise=0.28, random_state=2)
    rungs.append(("D3 noisy moons (non-linear)", x3, y3))

    # D4 — real: Wine, 13 features, 3 classes.
    wine = load_wine()
    rungs.append(("D4 Wine (real, 13-D, 3-class)", wine.data, wine.target))

    # D5 — real, harder: Breast Cancer, 30 features, class imbalance.
    bc = load_breast_cancer()
    rungs.append(("D5 Breast Cancer (real, 30-D)", bc.data, bc.target))

    return rungs


def clf_accuracy(build_and_predict, X, y):
    """Split, call build_and_predict(x_tr, y_tr, x_te) -> preds, return held-out accuracy."""
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    preds = build_and_predict(x_tr, y_tr, x_te)
    return accuracy_score(y_te, preds)



def _fit_predict(model, x_tr, y_tr, x_te):
    model.fit(x_tr, y_tr)
    return model.predict(x_te)


def _stump_adaboost(n_estimators=60, learning_rate=0.6):
    stump = DecisionTreeClassifier(max_depth=1, random_state=7)
    try:
        return AdaBoostClassifier(estimator=stump, n_estimators=n_estimators, learning_rate=learning_rate, random_state=7)
    except TypeError:
        return AdaBoostClassifier(base_estimator=stump, n_estimators=n_estimators, learning_rate=learning_rate, random_state=7)


def _project2d(X):
    X = np.asarray(X, dtype=float)
    if X.shape[1] == 1:
        return np.c_[X[:, 0], np.zeros(X.shape[0])]
    return X[:, :2]


def _plot_regions(ax, model, X, y, title):
    x2 = _project2d(X)
    scaler = StandardScaler()
    xs = scaler.fit_transform(x2)
    fitted = clone(model)
    try:
        fitted.fit(xs, y)
    except ValueError:
        fitted = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
        fitted.fit(xs, y)
    x_min = xs[:, 0].min() - 0.8
    x_max = xs[:, 0].max() + 0.8
    y_min = xs[:, 1].min() - 0.8
    y_max = xs[:, 1].max() + 0.8
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 80), np.linspace(y_min, y_max, 80))
    grid = np.c_[xx.ravel(), yy.ravel()]
    zz = fitted.predict(grid).reshape(xx.shape)
    ax.contourf(xx, yy, zz, alpha=0.25, cmap="tab10")
    ax.scatter(xs[:, 0], xs[:, 1], c=y, s=16, cmap="tab10", edgecolor="k", linewidth=0.2)
    ax.set_title(title, fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])


def summarize_ladder(rungs):
    for name, X, y in rungs:
        classes, counts = np.unique(y, return_counts=True)
        print(f"{name}: X={X.shape}, classes={dict(zip(classes.tolist(), counts.tolist()))}")
    sample_name, sample_X, sample_y = rungs[0]
    print("sample rung:", sample_name)
    print(np.c_[sample_X, sample_y][:4])


def run_ladder(build_and_predict):
    rows = []
    for i, (name, X, y) in enumerate(clf_ladder(), start=1):
        acc = clf_accuracy(build_and_predict, X, y)
        rows.append((i, name, float(acc)))
    print("rung | accuracy | dataset")
    for i, name, acc in rows:
        print(f"D{i} | {acc:.3f} | {name}")
    return rows


def plot_summary(rows, model_factory, title):
    rungs = clf_ladder()
    fig, axes = plt.subplots(2, 3, figsize=(13, 7))
    axes = axes.ravel()
    for ax, (name, X, y) in zip(axes[:5], rungs):
        _plot_regions(ax, model_factory(), X, y, name.split("(")[0])
    axes[5].plot([r[0] for r in rows], [r[2] for r in rows], marker="o")
    axes[5].set_ylim(0.0, 1.05)
    axes[5].set_xlabel("ladder rung")
    axes[5].set_ylabel("held-out accuracy")
    axes[5].set_title(title)
    axes[5].grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


def lesson_score(losses, cost, alternative):
    empirical = round(sum(losses) / len(losses), 3)
    score = round(empirical + cost, 3)
    gap = round(alternative - score, 3)
    return empirical, score, gap


def cost_sensitive_choice(raw_loss, cost, competitor):
    score = raw_loss + cost
    return score, score < competitor


def make_stack_model(cv=None, n_neighbors=7):
    estimators = [
        ("logreg", make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))),
        ("knn", KNeighborsClassifier(n_neighbors=n_neighbors)),
        ("tree", RandomForestClassifier(n_estimators=60, max_depth=4, random_state=7)),
    ]
    meta = LogisticRegression(max_iter=1000)
    if cv is None:
        cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=7)
    return StackingClassifier(estimators=estimators, final_estimator=meta, cv=cv, stack_method="predict_proba")


def tiny_blend_predict(x_tr, y_tr, x_te):
    models = [
        make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
        KNeighborsClassifier(n_neighbors=1),
        DecisionTreeClassifier(max_depth=1, random_state=7),
    ]
    probs = []
    for model in models:
        model.fit(x_tr, y_tr)
        probs.append(model.predict_proba(x_te))
    avg = np.mean(probs, axis=0)
    classes = models[0].classes_
    return classes[np.argmax(avg, axis=1)]


## The concept, built once (D1)

The lesson formula is $$\hat y=\sum_{m=1}^M a_m \hat y_m,\qquad \sum_m a_m=1$$. We first rebuild the tiny score: average the three lesson losses, add the cost, and assert the exact plan numbers.

In [ ]:

def stacking_blending_method():
    losses = np.array([0.191, 0.109, 0.437], dtype=float)
    empirical, score, gap = lesson_score(losses, 0.09, 0.384)
    assert empirical == 0.246
    assert score == 0.336
    assert gap == 0.048
    base_predictions = np.array([0.20, 0.70, 0.55])
    weights = np.array([0.25, 0.45, 0.30])
    blended = float(np.sum(weights * base_predictions))
    assert abs(weights.sum() - 1.0) < 1e-12
    return {"empirical": empirical, "score": score, "gap": gap, "blended_probability": blended}

result = stacking_blending_method()
print(result)


The printed dictionary contains the hand-checkable lesson score plus one method-specific quantity: a vote weight, an additive update, a second-order surrogate, a blend, a margin objective, or a kernel identity.

In [ ]:
checked = stacking_blending_method()
assert checked['score'] == 0.336
print('D1 arithmetic verified for 3.28')

## The dataset ladder

All classification notebooks use the shared `clf_ladder()` and `clf_accuracy()` helpers embedded above, so the notebook is self-contained in Colab.

In [ ]:
rungs = clf_ladder()
summarize_ladder(rungs)

## Run the same method across D1-D5

The metric is held-out accuracy. Macro-F1 would be a useful companion when class skew is severe, especially on D5.

In [ ]:


def make_stack_model(cv=None, n_neighbors=7):
    estimators = [
        ("logreg", make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))),
        ("knn", KNeighborsClassifier(n_neighbors=n_neighbors)),
        ("tree", RandomForestClassifier(n_estimators=60, max_depth=4, random_state=7)),
    ]
    meta = LogisticRegression(max_iter=1000)
    if cv is None:
        cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=7)
    return StackingClassifier(estimators=estimators, final_estimator=meta, cv=cv, stack_method="predict_proba")


def tiny_blend_predict(x_tr, y_tr, x_te):
    models = [
        make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
        KNeighborsClassifier(n_neighbors=1),
        DecisionTreeClassifier(max_depth=1, random_state=7),
    ]
    probs = []
    for model in models:
        model.fit(x_tr, y_tr)
        probs.append(model.predict_proba(x_te))
    avg = np.mean(probs, axis=0)
    classes = models[0].classes_
    return classes[np.argmax(avg, axis=1)]


def build_and_predict(x_tr, y_tr, x_te):
    if len(y_tr) < 12:
        return tiny_blend_predict(x_tr, y_tr, x_te)
    model = make_stack_model()
    return _fit_predict(model, x_tr, y_tr, x_te)

rows = run_ladder(build_and_predict)
assert len(rows) == 5
assert all(0.0 <= acc <= 1.0 for _, _, acc in rows)


## Results visualization

The small multiples show the learned artifact on the first two standardized features for every rung; the summary panel tracks accuracy as the data become more realistic.

In [ ]:
plot_summary(rows, lambda: make_stack_model(), 'Stacking & blending accuracy')

## Pitfall on D5: optimizing the raw term and forgetting the cost

On D5, using in-sample base predictions makes the stack look better than it is. The fix is honest out-of-fold stacking plus the lesson's cost-aware score. The wrong behavior ranks by raw validation loss; the fix ranks by the lesson score with cost and the validation-gap scale.

In [ ]:

X, y = clf_ladder()[-1][1:]
x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
scaler = StandardScaler()
x_tr = scaler.fit_transform(x_tr)
x_te = scaler.transform(x_te)
lean = make_stack_model()
large = make_stack_model()
if hasattr(large, "set_params"):
    params = large.get_params()
    if "n_estimators" in params:
        large.set_params(n_estimators=min(params["n_estimators"] * 3, 240))
    if "max_iter" in params and params["max_iter"] > 0:
        large.set_params(max_iter=min(params["max_iter"] * 3, 240))
    if "C" in params:
        large.set_params(C=25.0)
    if "gamma" in params:
        large.set_params(gamma=3.0)
lean.fit(x_tr, y_tr)
large.fit(x_tr, y_tr)
lean_loss = 1.0 - accuracy_score(y_te, lean.predict(x_te))
large_loss = 1.0 - accuracy_score(y_te, large.predict(x_te))
wrong_pick = "large" if large_loss <= lean_loss else "lean"
lean_score, lean_ok = cost_sensitive_choice(lean_loss, 0.09, large_loss + 0.09 + 0.048)
large_score = large_loss + 0.09 + 0.048
fixed_pick = "lean" if lean_score <= large_score else "large"
print("raw losses:", {"lean": round(lean_loss, 3), "large": round(large_loss, 3)})
print("wrong raw-loss pick:", wrong_pick)
print("cost-aware scores:", {"lean": round(lean_score, 3), "large": round(large_score, 3)})
print("fixed pick:", fixed_pick)
assert lean_score <= large_score or fixed_pick == "large"


## Evaluate it + Practice

- Compare held-out accuracy against a majority-class no-skill baseline.
- Sanity check that shuffling labels pushes accuracy toward chance.
- Ablate the key idea: fewer boosting rounds, no honest stacking, linear instead of kernel, or tiny/huge C should change the metric.
- Watch failure signals: unstable D5 score, perfect train accuracy with weak validation accuracy, or a cost-aware score that reverses the raw-loss winner.

Practice 1: change one hyperparameter and re-run the D1-D5 table.

Practice 2: add a majority-class baseline row for every rung.

Practice 3: repeat D5 with a different random split and compare the validation gap.